# [자사몰 소구점 분석]

In [2]:
from __future__ import annotations

import argparse
import csv
import hashlib
import json
import math
import random
import re
import sys
import time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple
from urllib.parse import urljoin

import pandas as pd
import requests
from bs4 import BeautifulSoup

---
---

## 1. 크롤링

https://www.phytonutri.kr/board/list.php?bdId=goodsreview  

- 공식몰 & 스마트 스토어 리뷰가 모두 있는 페이지  
- 스마트 스토어는 <네이버스마트스토어> 블럭으로 표시되어 있음  
<br>   

- 수집 리스트  
    - 리뷰 텍스트: filtered_message  
    - 평점: score  
    - 작성일: created_at  
    - 작성자: user_display_name  
    - 채널 구분: review_vendor ("smart_store" = 스마트스토어, null = 자사몰/고도몰 직접 작성)  
    - 상품명: product_name, 상품 URL: product_url  
    - 이미지 여부: media_count / images  
    - 페이지네이션: pagy.page, pagy.next (다음 페이지 없으면 null로 추정)  
<br>   

- 리뷰가 너무 많으므로, 2025년 1월부터 리뷰 분석으로 하려고 했으나  
    해당 페이지는 2026년 4월 22일부터의 리뷰까지만 크롤링 가능하여 리뷰 크롤링 기간이 제한됨  
    - 2026.04.22 ~ 2026.06.16  
    - 1페이지당 20개의 리뷰가 총 150페이지 있음 (리뷰 3,000개 취합)   
<br>  

- 분석 기간이 약 8주 정도로 짧아 한계 있음  
    - 계절성이나 장기 트렌드(예: 공구 시즌, 명절 프로모션 등)를 보기 어려움  
        → 최근 두 달간의 스냅샷 정도로 해석 범위 한정  

In [3]:
from crawl_crema_reviews import crawl_crema_reviews

df_test, info = crawl_crema_reviews(end_page=2, outdir="test_run")
print(info)
df_test[["score", "channel", "review_text", "created_date", "product_name"]].head(20)

page=1 status=ok rows=20
page=2 status=ok rows=20
{'stopped_reason': 'end_page(2) 도달', 'pages_crawled': 3, 'rows_total': 40, 'rows_after_date_filter': 40, 'channel_breakdown_before_filter': {'스마트스토어': 38, '자사몰(고도몰)': 2}, 'saved_full_csv': 'test_run\\crema_reviews_full.csv', 'saved_filtered_csv': 'test_run\\crema_reviews_full.csv'}
{'stopped_reason': 'end_page(2) 도달', 'pages_crawled': 3, 'rows_total': 40, 'rows_after_date_filter': 40, 'channel_breakdown_before_filter': {'스마트스토어': 38, '자사몰(고도몰)': 2}, 'saved_full_csv': 'test_run\\crema_reviews_full.csv', 'saved_filtered_csv': 'test_run\\crema_reviews_full.csv'}


,score,channel,review_text,created_date,product_name
0,5,스마트스토어,먹기 전후 차이가 날만큼 꼭 필요,2026-06-15,지니어스뉴 드롭스&톡캡스
1,5,스마트스토어,다먹고 또 재구매 합니다.,2026-06-15,지니어스뉴 드롭스&톡캡스
2,5,스마트스토어,한달리뷰 남깁니다.\n3병째 먹이고있어요~!! \n처음엔 아침 저녁으로 두번 나눠먹...,2026-06-15,지니어스뉴 드롭스&톡캡스
3,5,스마트스토어,첫번째사진 왼쪽은 비타민 오른쪽은 지니어스뉴!\n우리 아기 영양제 돌 지나고 나서부...,2026-06-15,지니어스뉴 + 투데이디3 세트
4,5,스마트스토어,아이에게 꾸준히 먹이고 있는 \n제품입니다~\n이것만 먹여보니 아이가 맛에 거부감이...,2026-06-15,"그로우뉴 아이 유기농칼슘&마그네슘 비타민D,k2,망간"
5,5,스마트스토어,믿고 먹는 제품이에요. 1살8개월부터 5개월째 먹이고 있어요. 우유에 타서 매일 잘...,2026-06-15,"그로우뉴 아이 유기농칼슘&마그네슘 비타민D,k2,망간"
6,5,스마트스토어,1년6개월째 이제품만 먹었어요. 꾸준히 매일 먹이고 있답니다. 아이는 밝고 건강해요...,2026-06-15,지니어스뉴 드롭스&톡캡스
7,5,스마트스토어,개별포장 되어있어서 먹이고 관리하기 너무 편해요~ 애용하고 있습니다!,2026-06-15,투데이디3 for baby
8,5,스마트스토어,양이 너무 적네요 얼마 안먹고 끝이에요 아쉬워요ㅠㅠ,2026-06-15,지니어스뉴 드롭스&톡캡스
9,5,스마트스토어,아기가 잘먹고있어요 !!!! 또 주문해야겠어요,2026-06-15,지니어스뉴 드롭스&톡캡스


In [4]:
df, info = crawl_crema_reviews(
    since_date="2025-01-01",
    outdir="phytonutri_crema_reviews_2025",
)
print(info)

page=1 status=ok rows=20
page=2 status=ok rows=20
page=3 status=ok rows=20
page=4 status=ok rows=20
page=5 status=ok rows=20
page=6 status=ok rows=20
page=7 status=ok rows=20
page=8 status=ok rows=20
page=9 status=ok rows=20
page=10 status=ok rows=20
page=11 status=ok rows=20
page=12 status=ok rows=20
page=13 status=ok rows=20
page=14 status=ok rows=20
page=15 status=ok rows=20
page=16 status=ok rows=20
page=17 status=ok rows=20
page=18 status=ok rows=20
page=19 status=ok rows=20
page=20 status=ok rows=20
  [checkpoint] page=20 누적 rows=400
page=21 status=ok rows=20
page=22 status=ok rows=20
page=23 status=ok rows=20
page=24 status=ok rows=20
page=25 status=ok rows=20
page=26 status=ok rows=20
page=27 status=ok rows=20
page=28 status=ok rows=20
page=29 status=ok rows=20
page=30 status=ok rows=20
page=31 status=ok rows=20
page=32 status=ok rows=20
page=33 status=ok rows=20
page=34 status=ok rows=20
page=35 status=ok rows=20
page=36 status=ok rows=20
page=37 status=ok rows=20
page=38 stat

In [5]:
print(info)
df.shape
df["channel"].value_counts()
df.head(10)

{'stopped_reason': 'pagy.next 없음 (마지막 페이지 도달)', 'pages_crawled': 150, 'rows_total': 3000, 'rows_after_date_filter': 3000, 'channel_breakdown_before_filter': {'스마트스토어': 2830, '자사몰(고도몰)': 170}, 'saved_full_csv': 'phytonutri_crema_reviews_2025\\crema_reviews_full.csv', 'saved_filtered_csv': 'phytonutri_crema_reviews_2025\\crema_reviews_since_2025-01-01.csv'}


,review_id,page,score,review_vendor_raw,channel,review_text,created_at,created_date,author_display_name,author_grade,...,product_code,product_name,product_url,product_meta_score,product_meta_reviews_count,likes_count,comments_count,ai_summary,product_options,customer_properties
0,98444,1,5,smart_store,스마트스토어,먹기 전후 차이가 날만큼 꼭 필요,2026-06-15T18:56:08+09:00,2026-06-15,네이버 스마트스토어 구****,5,...,1000000281,지니어스뉴 드롭스&톡캡스,https://www.phytonutri.kr/goods/goods_view.php...,4.9,14386,0,0,"{'1': ['만족도', '효과']}",[],[]
1,98445,1,5,smart_store,스마트스토어,다먹고 또 재구매 합니다.,2026-06-15T18:55:42+09:00,2026-06-15,네이버 스마트스토어 구****,5,...,1000000281,지니어스뉴 드롭스&톡캡스,https://www.phytonutri.kr/goods/goods_view.php...,4.9,14386,0,0,{'2': ['만족도']},[],[]
2,98446,1,5,smart_store,스마트스토어,한달리뷰 남깁니다.\n3병째 먹이고있어요~!! \n처음엔 아침 저녁으로 두번 나눠먹...,2026-06-15T17:55:32+09:00,2026-06-15,네이버 스마트스토어 구****,5,...,1000000281,지니어스뉴 드롭스&톡캡스,https://www.phytonutri.kr/goods/goods_view.php...,4.9,14386,0,0,"{'1': ['만족도'], '0': ['맛']}",[],[]
3,98447,1,5,smart_store,스마트스토어,첫번째사진 왼쪽은 비타민 오른쪽은 지니어스뉴!\n우리 아기 영양제 돌 지나고 나서부...,2026-06-15T16:07:50+09:00,2026-06-15,네이버 스마트스토어 구****,5,...,1000000326,지니어스뉴 + 투데이디3 세트,https://www.phytonutri.kr/goods/goods_view.php...,4.8,245,0,0,"{'1': ['만족도', '배송', '유통기한']}",[],[]
4,98448,1,5,smart_store,스마트스토어,아이에게 꾸준히 먹이고 있는 \n제품입니다~\n이것만 먹여보니 아이가 맛에 거부감이...,2026-06-15T15:04:00+09:00,2026-06-15,네이버 스마트스토어 구****,5,...,1000000297,"그로우뉴 아이 유기농칼슘&마그네슘 비타민D,k2,망간",https://www.phytonutri.kr/goods/goods_view.php...,4.9,4844,0,0,"{'1': ['만족도', '편의성'], '-1': ['맛']}",[],[]
5,98449,1,5,smart_store,스마트스토어,믿고 먹는 제품이에요. 1살8개월부터 5개월째 먹이고 있어요. 우유에 타서 매일 잘...,2026-06-15T14:44:28+09:00,2026-06-15,네이버 스마트스토어 구****,5,...,1000000297,"그로우뉴 아이 유기농칼슘&마그네슘 비타민D,k2,망간",https://www.phytonutri.kr/goods/goods_view.php...,4.9,4844,0,0,{'1': ['만족도']},[],[]
6,98450,1,5,smart_store,스마트스토어,1년6개월째 이제품만 먹었어요. 꾸준히 매일 먹이고 있답니다. 아이는 밝고 건강해요...,2026-06-15T14:43:14+09:00,2026-06-15,네이버 스마트스토어 구****,5,...,1000000281,지니어스뉴 드롭스&톡캡스,https://www.phytonutri.kr/goods/goods_view.php...,4.9,14386,0,0,"{'2': ['만족도'], '1': ['성분']}",[],[]
7,98451,1,5,smart_store,스마트스토어,개별포장 되어있어서 먹이고 관리하기 너무 편해요~ 애용하고 있습니다!,2026-06-15T14:06:53+09:00,2026-06-15,네이버 스마트스토어 구****,5,...,1000000325,투데이디3 for baby,https://www.phytonutri.kr/goods/goods_view.php...,4.9,3634,0,0,"{'1': ['만족도', '편의성']}",[],[]
8,98452,1,5,smart_store,스마트스토어,양이 너무 적네요 얼마 안먹고 끝이에요 아쉬워요ㅠㅠ,2026-06-15T14:02:19+09:00,2026-06-15,네이버 스마트스토어 구****,5,...,1000000281,지니어스뉴 드롭스&톡캡스,https://www.phytonutri.kr/goods/goods_view.php...,4.9,14386,0,0,"{'-1': ['만족도'], '-2': ['용량']}",[],[]
9,98453,1,5,smart_store,스마트스토어,아기가 잘먹고있어요 !!!! 또 주문해야겠어요,2026-06-15T14:00:17+09:00,2026-06-15,네이버 스마트스토어 구****,5,...,1000000281,지니어스뉴 드롭스&톡캡스,https://www.phytonutri.kr/goods/goods_view.php...,4.9,14386,0,0,{'1': ['만족도']},[],[]


- 리뷰 수  
    - 스마트스토어 2,830건(94%)  
    - 자사몰 170건(6%)  

---

## 2. 리뷰 분석

- 외부몰 리뷰 분석과 동일한 분류 로직 적용  
    - appeal_keywords(키 정밀도 수정 버전)  
    - split_sentences  
    - analyze_appeal_sentiment  
<br>  

- 출력물  
    - 통합표: 자사몰+스마트스토어 합쳐서 카테고리×감성  
    - 채널별 절대값: 스마트스토어/자사몰 따로, 카테고리×감성  
        (행 인덱스가 channel, category 2단계)  
    - 채널별 비율(%): 각 채널 내부에서 100% 기준으로 정규화한 비율.  
        표본 크기 차이(2,830 vs 170)를 보정해서,  
        "어느 채널이 어떤 카테고리에 더 민감한지"를 공정하게 비교할 수 있는 표  
<br>  

- 시크릿 링크 201건(전체 3,000건의 약 6.7%) 완전 제외  
    - 201건이 달린 product_code=1000000402의 product_meta_reviews_count가 22,605건으로,  
    단일 제품치고는 너무 커서 여러 제품을 한 페이지에서 같이 파는 통합 이벤트 SKU로 추정됨  
    - 리뷰 텍스트 기반 제품 추론을 시도했으나, 실제 7건 샘플 검증 결과 6/7건이 제품명을 언급하지 않는 일반적인 후기("잘 먹어요", "배송이 느렸어요")라 분류 불가능했음  
    - product_options 필드로 정확한 매핑을 확인하려 했으나 신뢰할 수 있는 근거를 확보하지 못함  

In [6]:
from analyze_crema_reviews_voc import (
    analyze_reviews_with_meta, build_summary_tables,
    build_product_summary, display_voc_tables
)

df = pd.read_csv("phytonutri_crema_reviews_2025/crema_reviews_full.csv")

long_df = analyze_reviews_with_meta(df)
combined, by_channel, by_channel_pct = build_summary_tables(long_df)

display_voc_tables(combined, by_channel, by_channel_pct)

# 제품별 상위 소구점 (리뷰 많은 상품 10개)
product_summary = build_product_summary(long_df, top_n_products=10)
product_summary

,긍정,부정,중립
카테고리,,,
효능/효과,97,0,102
맛/복용편의,370,41,112
성분/안전성,69,3,50
가격/가치,211,4,60
배송/포장/품질,180,5,125


category,효능/효과,맛/복용편의,성분/안전성,가격/가치,배송/포장/품질,리뷰수
product_name,,,,,,
지니어스뉴 드롭스&톡캡스,62,237,33,109,104,428
투데이디3 for baby,19,72,21,55,86,200
"그로우뉴 아이 유기농칼슘&마그네슘 비타민D,k2,망간",53,55,16,21,32,140
🎁가정의 달 맞이 시크릿 링크🎁 오직 이 링크에서만 전제품 할인,8,39,12,21,34,89
지니어스뉴 + 그로우뉴 세트,12,19,4,8,5,38
이트뮨 아이 배도라지즙&엘더베리사과주스,7,22,3,7,7,37
KD 파마 듀얼 초임계 rTG 오메가3,4,9,8,15,8,33
"그로우뉴 ~키즈 유기농 칼슘&마그네슘 비타민D,k2 망간",2,13,2,3,3,21
듀오좀 마그앤알티지 식물성 듀얼 오메가3,13,14,10,7,5,20


- 맛/복용편의 카테고리의 부정 비율이 두 채널 모두 압도적으로 높음  
    - 스마트스토어 83.5%, 자사몰 85.7%  
    - 다른 카테고리(가격/가치, 배송/포장/품질, 효능/효과)는 부정 비율이 0~14% 수준  
    - 맛/복용편의만 80%대로 튀는 건 "불만이 생기면 대부분 맛/복용편의 쪽 문제"라는 의미  
    - 이건 일관된 패턴이라, 채널 차이보다 제품 자체의 공통 약점으로 해석하는 게 맞을 듯  
<br>  

- 효능/효과는 두 채널 다 부정이 0%  
    - 효과가 없다는 불만은 거의 없고,  
    - 불만은 맛/섭취 편의성 쪽에 몰려있다는 의미  
<br>  

- 가격/가치는 스마트스토어 부정 6.3% vs 자사몰 0%로 약간 차이가 있음  
    - 절대 건수가 자사몰은 5건 중 0건이라 표본이 작아서 우연일 가능성도 있음  
<br>  

- 제품별로 보면 지니어스뉴(드롭스&톡캡스)가 리뷰 428건으로 가장 많음  
<br>  

- 맛/복용편의가 237건으로 카테고리 중 압도적 1위  
    - 전체 맛/복용편의 부정(72건)의 상당 부분을 차지하고 있을 가능성이 높아서,  
    - "맛/복용편의 불만 = 주로 지니어스뉴 얘기"인지 한 번 더 확인이 필요  
<br>  
<br>  

cf. 제품별 표에서 '🎁가정의 달 맞이 시크릿 링크🎁 오직 이 링크에서만 전제품 할인'이라는 항목이 리뷰 89건으로 4위에 올라와 있는데, 분석에서 별도 표시("프로모션/묶음상품")로 분류하는게 안전  
- 그대로 두면 전제품 할인이라는 제품의 소구점처럼 잘못 해석될 위험이 있음  
- 리뷰로 제품 분류하려고 했으나 한계가 있어 제외하기로 함

### 2-1. 지니어스뉴 부정 내용 재확인

In [7]:
from drilldown_genius_new_negative import drilldown_negative, keyword_frequency_in_negative

df = pd.read_csv("phytonutri_crema_reviews_2025/crema_reviews_full.csv")

neg_df = drilldown_negative(df, product_name="지니어스뉴 드롭스&톡캡스", category="맛/복용편의")
print(f"부정 문장 수: {len(neg_df)}건")

# 부정 문장에서 가장 많이 나오는 키워드
keyword_freq = keyword_frequency_in_negative(neg_df)
print(keyword_freq)

# 실제 부정 문장들 직접 확인 (이게 가장 중요한 부분)
pd.set_option("display.max_colwidth", 100)
neg_df[["channel", "score", "sentence", "created_date"]]

부정 문장 수: 22건
비린     16
비린내    12
거부감     4
냄새      4
맛       3
향       1
젤리      1
캡슐      1
dtype: int64


,channel,score,sentence,created_date
0,스마트스토어,5,비린내 난다는 분들 있어서 온도 실험해봤어요.,2026-06-14
1,스마트스토어,5,상온에서 하루 있으면 비린내 납니다.,2026-06-14
2,스마트스토어,5,그냥 먹기엔 아직 거부감 드나봐요 밥위에 우유 요거트 뿌려줍이당...,2026-06-13
3,스마트스토어,5,다른 오메가3는 비린내가 심해서 역했는데 여기껀 하나도 안나요,2026-06-11
4,스마트스토어,5,옷에 흘리면 비린내 끝장인데..,2026-06-08
5,스마트스토어,5,근데 비린내나서 ㅠ 좀 힘들어요 오매가먹고나면 ㅠ,2026-06-01
6,스마트스토어,4,뭔가 비린내는 아닌데 뭔가뭔가 냄새?가 나긴함,2026-05-29
7,스마트스토어,5,다만 비린 냄새가 좀 있어서 이유식에 넣어 먹고나면 비린냄새 빼기가 힘들긴해요😂,2026-05-28
8,스마트스토어,5,돌때부터 꾸준히 먹이고 있고 현재 17개월인데 여전히 안 비린지 숟가락에 주면 잘 받아먹어요!,2026-05-26
9,스마트스토어,5,​성분 배합도 믿음직스러운데 아이가 거부감은커녕 매일 먼저 찾아주니 부모 입장에서 이보다 편할 수가 없네요.,2026-05-22


- "비린" 키워드 보정 후에도 잘못 분류되고 있는 리뷰들이 보임   
    - 룰베이스 감성분석이 가진 근본적인 한계  
    - 총 22건의 부정 리뷰 중 8개의 잘못 분류된 건 제외  
    (3, 8, 9, 10, 15, 18, 20, 21번)  
<br>  

- 실제로 확실한 진짜 부정 11건  
1, 2, 4, 5, 6, 7, 13, 14, 16(평점2점), 17(평점3점), 19번  
    - 보관 온도 이슈 (1, 2번): 상온 보관 시 비린내 발생 → 보관 안내 부족일 수 있음  
    - 개체차/민감도 차이 (16, 19, 6번): 일부 성인/예민한 사람에게만 비린내 느껴짐  
    - 실사용 환경 문제 (4번): 옷에 묻으면 냄새가 강함 → 휴대/사용성 이슈  
    - 아기 거부 반응 (5, 13, 14, 7번): 비린내로 인해 아기가 뱉거나 거부  
    - 평점은 높아도 불만 존재 (16번 평점2, 17번 평점3 외엔 대부분 평점 5점)  
    → 별점 5점 안에 숨은 불만이라는 패턴이 여기서도 확인됨  
<br>  

- 비린내 자체는 경쟁사보다 약하다는 강한 긍정 신호 확인  
    그래도 일부 보관/개인차로 비린내가 나면 아기가 거부한다는 운영상 시사점이 남음  

### 2-2. 경쟁 제품군별 비교

- 자사몰 리뷰 데이터에는 커큐민이 빠져있어 제외  
- 경쟁사는 쿠팡으로 제한 (CJ온스타일은 텍스트 리뷰가 없어서 제외)
- 세트 상품은 리뷰 중복으로 넣어 처리  
<br>  
- 지니어스뉴 1255+36+117(세트)+39(세트)=1447,  
    그로우뉴 413+32+117(세트)=562,  
    투데이D3 509+47(세트)+39(세트)=595  

In [8]:
df["product_name"].value_counts()

product_name
지니어스뉴 드롭스&톡캡스                                      1255
투데이디3 for baby                                      509
그로우뉴 아이 유기농칼슘&마그네슘 비타민D,k2,망간                       413
🎁가정의 달 맞이 시크릿 링크🎁 오직 이 링크에서만 전제품 할인                 201
지니어스뉴 + 그로우뉴 세트                                     117
이트뮨 아이 배도라지즙&엘더베리사과주스                                81
데이프로바 for baby 아기 쌩쌩유산균                              63
KD 파마 듀얼 초임계 rTG 오메가3                                51
데이프로바 + 투데이디3 세트                                     47
지니어스뉴 + 투데이디3 세트                                     39
지니어스뉴 키즈 국내 유일 스퀴즈 짜먹는 어린이 오메가3 DHA ALA 콜린 올로메가      36
뉴스데일리베스트 X 파이토뉴트리 맨드로포즈+                             33
그로우뉴 ~키즈 유기농 칼슘&마그네슘 비타민D,k2 망간                      32
수드티 루이보스 허브 블렌드 티 강황 맘먼트 임산부 차                       32
리버티엑스                                                29
듀오좀 마그앤알티지 식물성 듀얼 오메가3                               27
옥토피아 X 파이토뉴트리 (PHYTONUTRI)                            7
에피베리어 곤약세라미드 1.8                   

In [9]:
PRODUCT_CATEGORY_MAP = {
    "센트휴 - 수용성 커큐민 바이오페인2X":                       "커큐민+",
    "순수채움 - 수용성 커큐민 맥시멈":                           "커큐민+",
    "가온담음 - 페라큐민 강황 수용성 커큐민":                     "커큐민+",
    "릴크리터스 - 어린이 오메가 3 DHA 라즈베리 레몬맛 구미":       "지니어스뉴 (오메가3)",
    "세노비스 - 키즈 츄어블 오메가3":                            "지니어스뉴 (오메가3)",
    "굿앤키즈 - 알티지 오메가3 츄어블":                          "지니어스뉴 (오메가3)",
    "건국유업 - 쑥쑥 키즈업 칼슘 마그네슘 아연 비타민D 츄어블":    "그로우뉴 (칼마디)",
    "비타민마을 - 맘편한 어린이 칼슘 마그네슘 아연 비타민D":        "그로우뉴 (칼마디)",
    "비타민마을 - 금쪽같은 내새끼 어린이 칼슘 마그네슘 비타민D":    "그로우뉴 (칼마디)",
}

In [10]:
from analyze_crema_reviews_voc import appeal_keywords, tag_text, split_sentences, sentence_sentiment, analyze_reviews_with_meta
from compare_self_vs_competitor import (
    build_self_brand_long_df, self_brand_summary, competitor_summary,
    build_side_by_side, display_comparison
)

df = pd.read_csv("phytonutri_crema_reviews_2025/crema_reviews_full.csv")
sentiment_df = pd.read_csv(r"C:\Users\jm\Desktop\JM_DA5_자료\(2026.04.27-)_아이펠톤\DATA\외부몰_경쟁제품\appeal_sentiment_result.csv")

# 1. 자사몰+스마트스토어 데이터를 경쟁사 분석 3개 카테고리로 매핑 + 소구점 분석
long_df_self = build_self_brand_long_df(df, analyze_reviews_with_meta)
self_pos, self_neg = self_brand_summary(long_df_self)

# 2. 쿠팡/CJ 경쟁사 분석 결과(sentiment_df, PRODUCT_CATEGORY_MAP)를 그대로 가져와서
comp_pos, comp_neg = competitor_summary(sentiment_df, PRODUCT_CATEGORY_MAP)

# 3. 나란히 비교 (비율 정규화)
compare_df = build_side_by_side(self_pos, self_neg, comp_pos, comp_neg, normalize=True, min_sample_size=10)
display_comparison(compare_df, is_pct=True)

compare_df_abs = build_side_by_side(self_pos, self_neg, comp_pos, comp_neg, normalize=False)
display_comparison(compare_df_abs, is_pct=False)

- 확실하게 쓸 수 있는 것 (OK 표시, 표본 12건 이상)  
    - 지니어스뉴 맛/복용편의: 자사 부정 10.5%(25/239) vs 경쟁사 부정 37.7%(23/61)  
        → 자사가 명확히 우위  
    - 그로우뉴 맛/복용편의: 자사 부정 5.9%(5/85) vs 경쟁사 부정 46.4%(26/56)  
        → 격차가 더 크게 자사 우위  
    - 지니어스뉴/그로우뉴 가격/가치, 효능/효과, 그로우뉴 성분/안전성:  
    모두 자사가 경쟁사보다 부정 비율이 낮음. 일관된 패턴  
    - 지니어스뉴 배송/포장/품질(16건): 자사 3.0% vs 경쟁사 68.8%  
        → 격차 매우 큼. 경쟁사 표본이 16건이라 OK 처리됐지만,  
        이 정도면 그래도 어느 정도 안정적인 수치로 볼 수 있음  
<br>  
- 참고용으로만 쓸 것 (노란색, 표본 10건 미만)  
<br>  

> 지니어스뉴와 그로우뉴 두 제품군 모두,  
> 표본이 충분한 맛/복용편의·가격/가치·효능/효과 항목에서 자사 리뷰의 부정 비율이 경쟁사보다 일관되게 낮다.  
> 
> 특히 맛/복용편의(자사 5~10% vs 경쟁사 38~46%)와 배송/포장/품질(자사 3% vs 경쟁사 69%, 단 후자는 표본 16건)에서 격차가 두드러진다.  